In [2]:
# 1. Instala a dependência zstd no sistema Ubuntu do Colab
!sudo apt-get update -y && sudo apt-get install -y zstd

# 2. Instala o Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Inicia o servidor do Ollama em segundo plano
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# 4. Baixa o modelo Qwen 2.5 Coder 7B
!ollama pull qwen2.5-coder:7b

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [108 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,908 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,314 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,202 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-sec

In [4]:
!pip install -q gradio requests duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 65.0 MB/s eta 0:00:00


In [ ]:
import os
import subprocess
import requests
import json
import gradio as gr
from duckduckgo_search import DDGS

# --- FUNÇÕES DE SUPORTE ---

def search_web(query):
    """Realiza busca em tempo real na web."""
    try:
        results = []
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=3):
                results.append(f"Título: {r['title']}\nResumo: {r['body']}\nURL: {r['href']}")
        return "\n\n".join(results)
    except Exception as e:
        return f"Erro na busca web: {str(e)}"

def process_file(file):
    """Lê o conteúdo de arquivos anexados no chat."""
    if file is None:
        return ""
    try:
        with open(file.name, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
        filename = os.path.basename(file.name)
        return f"\n\n--- Conteúdo do Arquivo Anexado ({filename}) ---\n{content}\n--- Fim do Arquivo ---\n"
    except Exception as e:
        return f"\n[Erro ao ler arquivo: {str(e)}]"

def clone_github_repo(repo_url):
    """Clona um repositório GitHub para a pasta local do Colab."""
    if not repo_url or not repo_url.strip():
        return "Nenhum repositório informado."
    try:
        repo_name = repo_url.strip().split("/")[-1].replace(".git", "")
        if os.path.exists(repo_name):
            return f"Repositório '{repo_name}' já existe no ambiente."

        result = subprocess.run(["git", "clone", repo_url.strip()], capture_output=True, text=True)
        if result.returncode == 0:
            return f"Sucesso! Repositório '{repo_name}' clonado no ambiente."
        else:
            return f"Erro ao clonar: {result.stderr}"
    except Exception as e:
        return f"Falha na execução: {str(e)}"

# --- LÓGICA DO AGENTE ---

def chat_response(message, history, file, enable_web_search, github_url):
    # System Prompt base definindo a personalidade e regras
    system_prompt = (
        "Você é um Agente Especialista em Engenharia de Software e Programação (Qwen 2.5 Coder). "
        "Você analisa repositórios, corrige bugs, escreve código limpo e responde sempre em Português do Brasil. "
        "Mantenha o contexto do histórico de conversa com o usuário."
    )

    extra_context = ""

    # 1. Processa repositório GitHub se fornecido
    if github_url and github_url.strip():
        clone_msg = clone_github_repo(github_url)
        extra_context += f"\n[Status do GitHub: {clone_msg}]\n"

    # 2. Processa arquivo anexado no chat
    if file:
        extra_context += process_file(file)

    # 3. Realiza busca na web se a caixa estiver marcada
    if enable_web_search:
        search_results = search_web(message)
        extra_context += f"\n\n--- Resultados da Busca na Web ---\n{search_results}\n--- Fim da Busca ---\n"

    # 4. Monta o histórico estruturado de mensagens
    messages = [{"role": "system", "content": system_prompt}]

    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        if bot_msg:
            messages.append({"role": "assistant", "content": bot_msg})

    full_user_content = message + extra_context
    messages.append({"role": "user", "content": full_user_content})

    # 5. Chamada para o Ollama
    url = "http://localhost:11434/api/chat"
    payload = {
        "model": "qwen2.5-coder:7b",
        "messages": messages,
        "stream": False
    }

    try:
        response = requests.post(url, json=payload)
        response_data = response.json()
        return response_data.get("message", {}).get("content", "Erro ao obter resposta da IA.")
    except Exception as e:
        return f"Erro na requisição ao servidor Ollama: {str(e)}"

# --- INTERFACE GRADIO (COMPATÍVEL COM A VERSÃO ATUAL) ---

with gr.Blocks(title="Agente Coder Completo") as demo:
    gr.Markdown("# 🤖 Agente Coder (Qwen 2.5) - GitHub, Arquivos & Busca Web")

    with gr.Row():
        github_input = gr.Textbox(
            label="Link do Repositório GitHub (Opcional)",
            placeholder="https://github.com/usuario/repositorio.git",
            scale=3
        )
        web_search_checkbox = gr.Checkbox(
            label="Ativar Busca na Web (DuckDuckGo)",
            value=False,
            scale=1
        )

    file_input = gr.File(
        label="Anexar Arquivo de Código / Texto (Opcional)",
        file_types=[".txt", ".py", ".js", ".html", ".css", ".json", ".csv", ".md", ".cpp", ".java", ".ts"]
    )

    chatbot = gr.ChatInterface(
        fn=chat_response,
        additional_inputs=[
            file_input,
            web_search_checkbox,
            github_input
        ]
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3203c884e9dfe790a6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
